In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## CVM - Fundos Investimentos - Registros Subclasse

In [0]:
bronze_path_registro_subclasse_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_subclasse_cvm/"

df_registro_subclasse_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_subclasse_cvm)

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropala 
colunas_obrigatorias = ['ID_Registro_Classe', 'ID_Subclasse', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_subclasse_cvm = df_registro_subclasse_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Em **registro_subclasse** só retiramos os fundos com ***Situacao*** de **Cancelado**

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm.filter(f.col("Situacao") != 'Cancelado')

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
# Dropando a Data de Processamento da Bronze 
df_registro_subclasse_cvm = df_registro_subclasse_cvm.drop("data_processamento")

# Criando a Data de Processamento da silver
df_registro_subclasse_cvm = df_registro_subclasse_cvm.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('id_subclasse', f.col('ID_Subclasse').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('previdenciario', f.col('Previdenciario').cast(t.StringType())) \
    .withColumn('exclusivo_inr', f.col('Exclusivo_INR').cast(t.StringType())) \
    .withColumn('exclusivo_previdencia_complementar', f.col('Exclusivo_Previdencia_Complementar').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


### 1.2 Salvar na camada Silver


In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_subclasse_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_subclasse_cvm")